# Задача на 4-ом шаге урока

**🛻 Как там твоя Tesla?**

In [150]:
!pip install woodwork -q
!pip install featuretools -q

In [151]:
import pandas as pd

import featuretools as ft
from woodwork.logical_types import Categorical, Double, Datetime, Age, Integer

In [152]:
data_root = "https://raw.githubusercontent.com/a-milenkin/Competitive_Data_Science/main/data/"

path = data_root + 'car_train.csv'
car_info = pd.read_csv(path)

path = data_root + 'rides_info.csv'
rides_info = pd.read_csv(path)

path = data_root + 'driver_info.csv'
driver_info = pd.read_csv(path)

path = data_root + 'fix_info.csv'
fix_info = pd.read_csv(path)

In [153]:
rides_info.head(3)

,user_id,car_id,ride_id,ride_date,rating,ride_duration,ride_cost,speed_avg,speed_max,stop_times,distance,refueling,user_ride_quality,deviation_normal
0,o52317055h,A-1049127W,b1v,2020-01-01,4.95,21,268,36,113.548538,0,514.246920,0,1.115260,2.909
1,H41298704y,A-1049127W,T1U,2020-01-01,6.91,8,59,36,93.000000,1,197.520662,0,1.650465,4.133
2,v88009926E,A-1049127W,g1p,2020-01-02,6.01,20,315,61,81.959675,0,1276.328206,0,2.599112,2.461


In [154]:
es = ft.EntitySet(id="car_data")

models_df = car_info[['model']].drop_duplicates().reset_index(drop=True)
models_df['model_id'] = range(len(models_df))
models_df['model_id'] = models_df['model_id'].astype(int)

df_with_model_id = car_info.merge(models_df, on='model', how='left')
df_with_model_id['model_id'] = df_with_model_id['model_id'].fillna(-1).astype(int)

es = es.add_dataframe(
    dataframe_name="cars",
    dataframe=df_with_model_id,
    index="car_id",
    logical_types={
        'car_id': Categorical,
        "car_type": Categorical,
        'fuel_type': Categorical,
        'model': Categorical,
        'year_to_start': Age,
        'year_to_work': Age,
        'car_rating': Double,
        'riders': Integer,
        # 'target_class': Categorical,
        # 'target_reg': Double
    }
)

es = es.add_dataframe(
    dataframe_name="rides",
    dataframe=rides_info.drop(['ride_id'], axis=1),
    index='index',
    time_index="ride_date",
    logical_types={
        'user_id': Categorical,
        'car_id': Categorical,
        'speed_max': Double
      }
    )

es = es.add_dataframe(
    dataframe_name="drivers",
    dataframe=driver_info,
    index="user_id",
    logical_types={
        "sex": Categorical,
        "first_ride_date": Datetime,
        "age": Age
      }
    )

es = es.add_dataframe(
    dataframe_name="fixes",
    dataframe=fix_info,
    index="index",
    logical_types={"work_type": Categorical, "worker_id":Categorical}
    )

es = es.add_dataframe(
    dataframe_name="models",
    dataframe=models_df,
    index="model_id",
    logical_types={
        'model': Categorical,
        'model_id': Integer
    }
)

es = es.add_relationship("cars", "car_id", "rides", "car_id")
es = es.add_relationship("drivers", "user_id", "rides", "user_id")
es = es.add_relationship("cars", "car_id", "fixes", "car_id")
es = es.add_relationship("models", "model_id", "cars", "model_id")

/usr/local/lib/python3.12/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/usr/local/lib/python3.12/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/usr/local/lib/python3.12/dist-packages/featuretools/entityset/entityset.py:1733: UserWarning: index index not found in dataframe, creating new integer column
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/woodwork/type_sys/utils.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pd.to_datetime(
/usr/

In [155]:
feature_matrix, feature_defs = ft.dfs(
    entityset=es,
    target_dataframe_name="models",
    agg_primitives=["sum", "count", "max"],
    trans_primitives=[],
    max_depth=2,
    verbose=True
)

Built 84 features
Elapsed: 00:01 | Progress:  42%|████▏     

/usr/local/lib/python3.12/dist-packages/featuretools/computational_backends/feature_set_calculator.py:785: FutureWarning: The provided callable <function sum at 0x7dbdcfa5c540> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  ).agg(to_agg)
/usr/local/lib/python3.12/dist-packages/featuretools/computational_backends/feature_set_calculator.py:785: FutureWarning: The provided callable <function max at 0x7dbdcfa5cc20> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "max" instead.
  ).agg(to_agg)
/usr/local/lib/python3.12/dist-packages/featuretools/computational_backends/feature_set_calculator.py:785: FutureWarning: The provided callable <function max at 0x7dbdcfa5cc20> is currently using SeriesGroupBy.max. In a future version of pandas, the provided callable will b

Elapsed: 00:01 | Progress: 100%|██████████


In [156]:
pd.set_option('display.max_columns', None)
feature_matrix_tesla = feature_matrix[feature_matrix['model'] == 'Tesla Model 3']

feature_matrix_tesla[
    [
        'COUNT(cars)',
        'MAX(rides.speed_max)',
    ]
]

,COUNT(cars),MAX(rides.speed_max)
model_id,,
6,14,199.370103


In [157]:
total_fixes = feature_matrix_tesla['COUNT(fixes)'].values[0]
total_cars = feature_matrix_tesla['COUNT(cars)'].values[0]
avg_fixes = total_fixes / total_cars
print(f"Среднее число ремонтов: {round(avg_fixes, 1)}")

Среднее число ремонтов: 34.4


In [158]:
# Максимальная суммарная длительность ремонтов на одну машину
max_repair_duration = feature_matrix_tesla['MAX(cars.SUM(fixes.work_duration))'].values[0]
print(f"Макс. суммарная длительность ремонта: {max_repair_duration}")

Макс. суммарная длительность ремонта: 970.0


# Задача на 7-ом шаге урока

**🤩 Выбираем себе дом в LA 🌴**

In [159]:
# dataset california housing

In [170]:
!pip install geopandas -q
import geopandas as gpd
from sklearn.datasets import fetch_california_housing

In [171]:
df = fetch_california_housing(as_frame=True).data
df.head(3)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24


In [172]:
# создадим GeoDataFrame
gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df['Longitude'], df['Latitude']),
        crs=4326
    ).to_crs(epsg=3857)
gdf.head(3)

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,geometry
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,POINT (-13606581.36 4562487.679)
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,POINT (-13605468.165 4559667.342)
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,POINT (-13607694.555 4558257.461)


In [173]:
import requests, zipfile, io

county_fname = 'https://github.com/a-milenkin/Competitive_Data_Science/raw/main/data/ca-county-boundaries.zip'
r2 = requests.get(county_fname)
z2 = zipfile.ZipFile(io.BytesIO(r2.content))
z2.extractall("./ca")
ca_counties=gpd.read_file('./ca/CA_Counties').to_crs("EPSG:3857")
gdf = gpd.overlay(gdf, ca_counties[['NAME', 'geometry']], how='intersection')
gdf.head(3)


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,NAME,geometry
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,Alameda,POINT (-13606581.36 4562487.679)
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,Alameda,POINT (-13605468.165 4559667.342)
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,Alameda,POINT (-13607694.555 4558257.461)


In [164]:
# Максимальное расстояние до центра LA

# Минимальное расстояние до центра LA

# Среднее расстояние до центра LA

# Задача на 8-ом шаге урока

**🤯 Не курс, а Санта-Барбара какая-то...**

Для отладки воспользуйтесь примером и датасетом (ca_counties) из демонстрационного ноутбука (или из прошлой задачи):

In [165]:
ca_counties.head(3)

,STATEFP,COUNTYFP,COUNTYNS,GEOID,NAME,NAMELSAD,LSAD,CLASSFP,MTFCC,CSAFP,CBSAFP,METDIVFP,FUNCSTAT,ALAND,AWATER,INTPTLAT,INTPTLON,geometry
0,06,091,00277310,06091,Sierra,Sierra County,06,H1,G4020,None,None,None,A,2468694587,23299110,+39.5769252,-120.5219926,"POLYGON ((-13431319.751 4821511.426, -13431312..."
1,06,067,00277298,06067,Sacramento,Sacramento County,06,H1,G4020,472,40900,None,A,2499183617,76073827,+38.4500114,-121.3404409,"POLYGON ((-13490651.476 4680831.603, -13490511..."
2,06,083,00277306,06083,Santa Barbara,Santa Barbara County,06,H1,G4020,None,42200,None,A,7084000598,2729814515,+34.5370572,-120.0399729,"MULTIPOLYGON (((-13423116.772 4042044.149, -13..."


In [166]:
# Заполните словарь params верными параметрами и их значениями.
# Датасет ca_counties уже записан в переменную ca_counties.

params_1 = {'geo_data': ca_counties,
            '?': ca_counties,
            'columns': ['?', '?'],
            '?': 'feature.properties.NAME',
            '?': '?',
            '?': 'YlGn'}

params_2 = {'?':[34.4208, -119.698],
            '?':5,
            'color':'?',
            '?':'?'}

In [167]:
import folium

m = folium.Map([37, -122], tiles='OpenStreetMap', zoom_start=6)

# наносим на карту округа с заливкой по площади водного пространства
folium.Choropleth(**params_1).add_to(m)

# наносим на карту город Санта-Барбара
folium.CircleMarker(**params_2).add_to(m)

m

ValueError: CircleMarker location must be assigned when added directly to map.

# Задача на 13-ом шаге урока

**Модифицируйте MinimalFCParameters из TSFresh**

In [168]:
!pip install tsfresh -q

In [169]:
# Посмотрим на набор признаков в MinimalFCParameters
from tsfresh.feature_extraction import MinimalFCParameters